# 03 · Archetype Discovery (descriptive)

**Status from EXP-04:** no k passed the pre-registered selection rule. The clusterings below are **descriptive partitions**, not validated archetypes.

- k = 2: the only stable role split (bootstrap ARI 0.756)
- k = 5: the near-miss (ARI 0.591 vs 0.6 cutoff)

Setup: fully evolved Pokémon (568), relative stats (stat / BST).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

from src.features.preprocessing import STAT_COLS, load_pokemon, make_features, select_subset
from src.models.clustering import assignment_margins, cluster_profiles, fit_kmeans

sns.set_theme(style="whitegrid")
data = select_subset(load_pokemon(), "fully_evolved")
X, _ = make_features(data, "relative")
print(X.shape)

## 1. Cluster profiles

Centroid values are **shares of total stats** (0.20 = 20%). An even spread would be 1/6 ≈ 0.167 for every stat.

`borderline_share` = fraction of members almost halfway to another cluster (margin < 10%). High values explain low stability.

In [ ]:
models, profiles = {}, {}
for k in (2, 5):
    models[k] = fit_kmeans(X, k)
    profiles[k] = cluster_profiles(data, X, models[k])
    print(f"\n===== k = {k} =====")
    display(profiles[k].round(3))

## 2. How each cluster differs from the average Pokémon

Bars show centroid share minus the overall average share, in percentage points. Positive = this group puts more of its stat budget there than typical.

In [ ]:
overall = X.mean()
fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 5]})
for ax, k in zip(axes, (2, 5)):
    diff = (profiles[k].set_index("cluster")[STAT_COLS] - overall) * 100
    diff.plot.bar(ax=ax, legend=(k == 5))
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"k = {k}: centroid minus average (percentage points)")
    ax.set_ylabel("pp of total stats")
if axes[1].get_legend():
    axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()

## 3. PCA: a 2D picture of 6D data

PCA finds new axes (principal components) that are mixes of the original stats, ordered by how much of the spread they capture. We plot the first two.

- **Clustering was done on all 6 features.** PCA is only for viewing.
- Points that overlap in 2D may be far apart in 6D, and vice versa. Check the explained variance: whatever the first two components miss is invisible in the plot.
- Relative stats sum to 1, so one direction has zero spread: the 6th component should explain ~0%.

In [ ]:
pca = PCA(random_state=42).fit(X)
print("Explained variance by component:", (pca.explained_variance_ratio_ * 100).round(1), "%")
print(f"First two components together: {pca.explained_variance_ratio_[:2].sum():.0%}")

loadings = pd.DataFrame(pca.components_[:2].T, index=STAT_COLS, columns=["PC1", "PC2"]).round(2)
print("\nLoadings (how much each stat feeds each axis):")
display(loadings)

In [ ]:
coords = pca.transform(X)[:, :2]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, k in zip(axes, (2, 5)):
    m = assignment_margins(X, models[k])
    sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=m["cluster"].to_numpy(), palette="tab10",
                    style=m["borderline"].map({True: "borderline", False: "clear"}).to_numpy(),
                    alpha=0.7, s=35, ax=ax)
    ax.set_title(f"k = {k} on PCA axes (x markers = borderline)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

# label a few extreme Pokémon so the axes are easy to read
for i in np.argsort(np.abs(coords).sum(axis=1))[-8:]:
    for ax in axes:
        ax.annotate(data["name"].iloc[i], coords[i], fontsize=8)
plt.tight_layout()

## 4. Your notes

(Write here: what each k=5 group looks like, whether k=2 is physical vs special, and why you think the groups were unstable.)